# Promover Seeds — Landing → Bronze → Silver

Promove as 6 tabelas de dimensão fixa (seeds) da Landing Zone para Bronze e Silver. Diferente das 11 tabelas de evento diário, seeds não têm sujeira intencional — só cast de tipo, sem UDFs de limpeza. Gravação por `overwrite`, não `MERGE` — o catálogo inteiro é recarregado a cada execução, não incremental.

Cada seed promovido é registrado em `observability.pipeline_runs` (ADR-014, Fase C).

Referências: ADR-001 (Landing Zone), ADR-013 (transformação, adaptada aqui para o caso mais simples), ADR-014.

In [0]:
import sys

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

if "/files/" in notebook_path:
    root_relative = notebook_path.split("/files/")[0] + "/files"
else:
    root_relative = notebook_path.rsplit("/src/", 1)[0]

candidatos = {root_relative}
if root_relative.startswith("/Workspace"):
    candidatos.add(root_relative[len("/Workspace"):])
else:
    candidatos.add("/Workspace" + root_relative)

for candidato in candidatos:
    if candidato not in sys.path:
        sys.path.append(candidato)

print("Candidatos adicionados ao sys.path:", candidatos)

In [0]:
# Promovendo os 6 seeds via função genérica, com registro de execução
from src.transformacao.configuracao_seeds import CONFIGURACAO_SEEDS
from src.transformacao.promover_seed import promover_seed
from src.observabilidade.registrar_execucao import registrar_execucao_pipeline

for tabela, config in CONFIGURACAO_SEEDS.items():
    resultado = promover_seed(spark=spark, tabela=tabela, config=config)
    print(resultado)

    registrar_execucao_pipeline(
        spark=spark,
        pipeline="promover_seeds",
        item=tabela,
        status=resultado["status"],
        detalhes=resultado,
    )

## Execução

Promove os 6 seeds via função genérica (`promover_seed`), guiada por `CONFIGURACAO_SEEDS`. Cada seed é registrado em `observability.pipeline_runs`.

In [0]:
df_runs = spark.table("poc_pulse_observability.observability.pipeline_runs")
df_runs.filter(df_runs.pipeline == "promover_seeds").show(10, truncate=False)